# Topic 4: APIs, JSON and XML Parsing## Collecting Structured Data from Web ServicesThis notebook is the practical companion for Topic 4 of the Unstructured Data module. In Topic 3 you learned to scrape data from web pages — extracting structure from HTML that was designed for humans. This topic takes a different approach: working with **APIs** (Application Programming Interfaces), which are designed to give programs direct, structured access to data.### How This Notebook Is Organised| Section | Mode | What You Will Learn ||---------|------|-------------------|| Part 1: What Are APIs? | 🏫 In-lesson | REST concepts, JSON format, APIs vs web scraping || Part 2: Working with a Real API | 🏫 In-lesson | Open-Meteo weather API, query parameters, handling responses || Part 3: From JSON to DataFrame | 🏫 In-lesson | Parsing nested JSON, building DataFrames, saving & loading || Part 4: XML Parsing | 🏫 In-lesson | ElementTree, navigating XML, JSON vs XML comparison || Part 5: Authentication & Best Practices | 📖 Independent study | API keys, rate limiting, pagination, error handling || Part 6: End-to-End Pipeline | 📖 Independent study | ONS API → structure → analyse (mirrors assessment) || Exercises | 📖 Independent study | Assessment-aligned consolidation tasks |### Link to Your AssessmentThis topic directly supports **Learning Outcome 2**: *Process unstructured and semi-structured data sources and reformat into more machine-readable formats.* The assessment rubric rewards clear understanding of data processing, proper error handling, and the ability to convert between data formats (JSON, XML, DataFrames). It also supports **Learning Outcome 4** (Python for data manipulation) and **Learning Outcome 5** (reproducible analysis and robust code).---

## SetupRun the cell below to install the required libraries:- **requests** — making HTTP requests to APIs- **pandas** — structuring API responses into DataFrames- **matplotlib** — visualising data from APIs

In [ ]:
%pip install --quiet requests pandas matplotlib

In [ ]:
import requests
import pandas as pd
import json
import time
import xml.etree.ElementTree as ET
from urllib.parse import urlencode, urlparse, parse_qs

---# 🏫 Part 1: What Are APIs?*Approximate time: 15 minutes*## 1.1 From Web Scraping to APIsIn Topic 3, you learned to extract data from websites by parsing HTML — a format designed for browsers and human readers. This works, but it has limitations:- HTML structures change without warning, breaking your scraper- You have to work around formatting, adverts, and navigation elements- You cannot easily ask the server for *specific* data — you get the whole page**APIs** solve these problems. An API is a structured interface that a service provides specifically for programs to request data. Instead of parsing messy HTML, you get clean, structured data — usually in **JSON** or **XML** format.### Scraping vs APIs: When to Use Which| | Web Scraping | APIs ||---|---|---|| **Data format** | HTML (unstructured) | JSON/XML (structured) || **Stability** | Fragile — breaks if the page changes | Stable — versioned and documented || **Authentication** | Usually none (but check ToS) | Often requires an API key || **Rate limits** | Implicit (be respectful) | Explicit (documented) || **Best for** | Sites with no API available | Services that offer a public API |> 💡 **Rule of thumb:** If an API exists, use the API. Only scrape when there is no API alternative.## 1.2 How APIs Work — A Quick RecapAPIs use the same HTTP protocol you worked with in Topic 3. The difference is in what you get back:```Topic 3 (Scraping):    GET http://books.toscrape.com/  →  HTML (for humans)Topic 4 (APIs):        GET https://api.open-meteo.com/ →  JSON (for programs)```The request-response cycle is identical:1. Your code sends an **HTTP GET request** with query parameters specifying what data you want2. The API server processes your request3. The server returns a **JSON or XML response** with the requested dataThe key difference is that API responses are *already structured* — you do not need to parse HTML trees or find patterns in page layouts.## 1.3 Understanding JSON**JSON** (JavaScript Object Notation) is the most common API response format. If you have worked with Python dictionaries and lists, you already understand JSON — the syntax is almost identical:```json{    "name": "London",    "latitude": 51.5074,    "longitude": -0.1278,    "population": 8982000,    "tags": ["capital", "financial centre", "historic"]}```| JSON Type | Python Equivalent | Example ||-----------|-------------------|---------|| Object `{}` | `dict` | `{"key": "value"}` || Array `[]` | `list` | `[1, 2, 3]` || String | `str` | `"hello"` || Number | `int` / `float` | `42`, `3.14` || Boolean | `bool` | `true` / `false` || Null | `None` | `null` |> 💼 **Assessment connection:** The assessment brief specifically mentions "converting JSON/XML into a structured data format like a Pandas DataFrame." Understanding these data types and how they map to Python is the foundation for that conversion.

---# 🏫 Part 2: Working with a Real API*Approximate time: 25 minutes*We will use the [Open-Meteo API](https://open-meteo.com/) — a free weather forecast API that requires no registration or API key. This makes it perfect for learning, and the "get the forecast for your area" angle makes it immediately tangible.## 2.1 Your First API RequestLet's fetch the hourly temperature forecast for London:

In [ ]:
url = "https://api.open-meteo.com/v1/forecast"

# Query parameters tell the API exactly what data we want
params = {
    "latitude": 51.5074,
    "longitude": -0.1278,
    "hourly": "temperature_2m"
}

response = requests.get(url, params=params)

print(f"Status code: {response.status_code}")
print(f"Content type: {response.headers.get('Content-Type', 'unknown')}")
print(f"Response size: {len(response.text):,} characters")

Notice how we pass query parameters as a dictionary rather than building the URL by hand. The `requests` library constructs the full URL for us:```https://api.open-meteo.com/v1/forecast?latitude=51.5074&longitude=-0.1278&hourly=temperature_2m```This is cleaner, less error-prone, and easier to modify than string concatenation.### Understanding the URL StructureLet's break down the URL to see how API endpoints are organised:

In [ ]:
# Parse the full URL that requests built for us
full_url = response.url
parsed = urlparse(full_url)

print(f"Full URL: {full_url}")
print(f"  Scheme:   {parsed.scheme}")        # https
print(f"  Host:     {parsed.hostname}")       # api.open-meteo.com
print(f"  Path:     {parsed.path}")           # /v1/forecast
print(f"  Query:    {parse_qs(parsed.query)}")  # the parameters

The path `/v1/forecast` tells us two things:- **v1** — this is version 1 of the API (versioning helps APIs evolve without breaking existing code)- **forecast** — this is the forecast endpoint (APIs often have multiple endpoints for different data)## 2.2 Parsing the JSON ResponseThe response body is a JSON string. We can convert it to a Python dictionary with `.json()`:

In [ ]:
data = response.json()

# Let's see what keys are at the top level
print("Top-level keys:")
for key in data.keys():
    value = data[key]
    if isinstance(value, dict):
        print(f"  {key}: dict with {len(value)} keys → {list(value.keys())[:5]}")
    elif isinstance(value, list):
        print(f"  {key}: list with {len(value)} items")
    else:
        print(f"  {key}: {value}")

### Navigating Nested JSONAPI responses are often **nested** — dictionaries within dictionaries, lists within dictionaries. The key skill is navigating this tree to find the data you need.Let's explore the structure step by step:

In [ ]:
# The hourly data is nested under the "hourly" key
hourly = data["hourly"]

print(f"Keys in 'hourly': {list(hourly.keys())}")
print(f"Number of time entries: {len(hourly['time'])}")
print(f"Number of temperature entries: {len(hourly['temperature_2m'])}")

# Show the first 5 entries
print(f"\nFirst 5 times: {hourly['time'][:5]}")
print(f"First 5 temps: {hourly['temperature_2m'][:5]}")

The JSON response has a parallel structure: the `time` array and `temperature_2m` array are aligned by index. Entry 0 in both arrays corresponds to the same hour.## 2.3 Requesting Different DataOne of the great strengths of APIs is that you can request exactly the data you need by changing the query parameters. Let's ask for more weather variables:

In [ ]:
# Request multiple weather variables
params_extended = {
    "latitude": 51.5074,
    "longitude": -0.1278,
    "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m"],
    "forecast_days": 3
}

response = requests.get(url, params=params_extended)
data_extended = response.json()

# See what we got back
print("Hourly variables returned:")
for key in data_extended["hourly"]:
    values = data_extended["hourly"][key]
    if key != "time":
        print(f"  {key}: {len(values)} values, first 3 = {values[:3]}")

### Try It YourselfChange the latitude and longitude to your location (or your workplace location) and run the cell again. You can find coordinates by searching "latitude longitude [your city]" or using [latlong.net](https://www.latlong.net/).This is a powerful pattern: **the same code, different parameters, different data**. Your assessment can use this approach — build a reusable function that accepts parameters and returns structured data from any API.---

---# 🏫 Part 3: From JSON to DataFrame*Approximate time: 20 minutes*## 3.1 Building a DataFrame from API DataThe most common task in data science is converting an API response into a pandas DataFrame for analysis. Let's convert our weather data:

In [ ]:
# Create a DataFrame from the hourly data
df_weather = pd.DataFrame({
    "time": pd.to_datetime(data_extended["hourly"]["time"]),
    "temperature_c": data_extended["hourly"]["temperature_2m"],
    "humidity_pct": data_extended["hourly"]["relative_humidity_2m"],
    "wind_speed_kmh": data_extended["hourly"]["wind_speed_10m"],
})

print(f"Shape: {df_weather.shape[0]} rows × {df_weather.shape[1]} columns")
print(f"Date range: {df_weather['time'].min()} to {df_weather['time'].max()}")
df_weather.head(10)

### Making It ReusableLet's wrap this into a function — the same principle we used in Topic 3 with `scrape_page()`. Writing reusable functions is a core professional practice that the assessment rewards:

In [ ]:
def get_weather_forecast(
    latitude: float,
    longitude: float,
    variables: list[str] = ["temperature_2m"],
    forecast_days: int = 7
) -> pd.DataFrame:
    """
    Fetch weather forecast data from the Open-Meteo API.

    Parameters
    ----------
    latitude : float
        Location latitude (e.g. 51.5074 for London)
    longitude : float
        Location longitude (e.g. -0.1278 for London)
    variables : list of str
        Weather variables to request (e.g. ["temperature_2m", "wind_speed_10m"])
    forecast_days : int
        Number of forecast days (1-16)

    Returns
    -------
    pd.DataFrame
        DataFrame with time index and requested weather variables
    """
    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": latitude,
            "longitude": longitude,
            "hourly": variables,
            "forecast_days": forecast_days,
        },
        timeout=10
    )
    response.raise_for_status()

    data = response.json()
    hourly = data["hourly"]

    df = pd.DataFrame({"time": pd.to_datetime(hourly["time"])})
    for var in variables:
        df[var] = hourly[var]

    return df

In [ ]:
# Use the function to get a 3-day forecast for London
df_london = get_weather_forecast(51.5074, -0.1278, ["temperature_2m", "wind_speed_10m"], 3)
print(f"London forecast: {len(df_london)} hourly readings")
df_london.head()

## 3.2 A Quick VisualisationLet's plot the temperature forecast to confirm our data looks sensible:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_london["time"], df_london["temperature_2m"], color="#2C7DA0", linewidth=1.5)
ax.set_title("London Temperature Forecast", fontsize=14)
ax.set_xlabel("Date/Time")
ax.set_ylabel("Temperature (°C)")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3.3 Saving and Loading JSONYou will often want to save API responses locally — for reproducibility, to avoid hitting rate limits, or to work offline. Python's built-in `json` module handles this:

In [ ]:
# Save the raw API response to a file
with open("weather_forecast.json", "w") as f:
    json.dump(data_extended, f, indent=2)

print("Saved to weather_forecast.json")

In [ ]:
# Load it back later
with open("weather_forecast.json", "r") as f:
    loaded_data = json.load(f)

# Verify it's the same data
print(f"Keys: {list(loaded_data.keys())}")
print(f"Temperatures match: {loaded_data['hourly']['temperature_2m'][:3] == data_extended['hourly']['temperature_2m'][:3]}")

In [ ]:
# Pretty-print JSON for inspection
# json.dumps() converts a Python object to a JSON *string* (note the 's' for string)
# json.dump() writes directly to a *file*
print(json.dumps(loaded_data["hourly_units"], indent=2))

### dump vs dumps, load vs loadsA common source of confusion:| Function | Purpose | Works with ||----------|---------|------------|| `json.dump()` | Write Python → JSON | **file** || `json.dumps()` | Write Python → JSON | **string** || `json.load()` | Read JSON → Python | **file** || `json.loads()` | Read JSON → Python | **string** |The `s` stands for "string". Without the `s`, the function works with files.> 💼 **Assessment connection:** Saving API responses to JSON files is good practice for your coursework. It means your analysis is reproducible — the assessor can see exactly what data you worked with, even if the API returns different data later.---

---# 🏫 Part 4: XML Parsing*Approximate time: 20 minutes*## 4.1 What Is XML?**XML** (eXtensible Markup Language) is an older data format that you will still encounter in many business contexts — especially in government data, financial services, healthcare, and legacy enterprise systems. It looks similar to HTML but is designed to carry data, not display it.Here is the same weather data expressed in JSON and XML:**JSON:**```json{    "location": "London",    "temperature": 15.2,    "conditions": ["cloudy", "mild"]}```**XML:**```xml<weather>    <location>London</location>    <temperature unit="celsius">15.2</temperature>    <conditions>        <condition>cloudy</condition>        <condition>mild</condition>    </conditions></weather>```### Key Differences| Feature | JSON | XML ||---------|------|-----|| **Syntax** | Curly braces and brackets | Opening and closing tags || **Attributes** | Not supported | Supported (e.g. `unit="celsius"`) || **Data types** | Yes (string, number, boolean) | No — everything is text || **Readability** | More concise | More verbose || **Modern APIs** | Standard | Less common, but still used |## 4.2 Parsing XML with ElementTreePython's built-in `xml.etree.ElementTree` module lets you parse and navigate XML documents. Let's work with a realistic example — an RSS news feed, which is XML:

In [ ]:
# Let's parse some XML — we'll create a realistic sample first
# This represents a simplified RSS feed structure
sample_xml = """<?xml version="1.0" encoding="UTF-8"?>
<rss version="2.0">
    <channel>
        <title>Tech Industry News</title>
        <description>Latest technology news</description>
        <item>
            <title>New Python Release Improves Performance</title>
            <link>https://example.com/python-release</link>
            <pubDate>Mon, 10 Feb 2026 09:00:00 GMT</pubDate>
            <category>Programming</category>
            <description>Python 3.14 delivers significant speed improvements.</description>
        </item>
        <item>
            <title>AI Regulation Framework Published</title>
            <link>https://example.com/ai-regulation</link>
            <pubDate>Sun, 09 Feb 2026 14:30:00 GMT</pubDate>
            <category>Policy</category>
            <description>The UK government publishes new AI safety guidelines.</description>
        </item>
        <item>
            <title>Cloud Computing Costs Fall for Third Quarter</title>
            <link>https://example.com/cloud-costs</link>
            <pubDate>Sat, 08 Feb 2026 11:15:00 GMT</pubDate>
            <category>Cloud</category>
            <description>Major providers reduce pricing across compute services.</description>
        </item>
    </channel>
</rss>
"""

# Parse the XML string
root = ET.fromstring(sample_xml)

print(f"Root tag: {root.tag}")
print(f"Root attributes: {root.attrib}")

### Navigating the XML TreeXML documents are trees, just like HTML. ElementTree provides methods to find and iterate over elements:

In [ ]:
# Find the channel title
channel = root.find("channel")
channel_title = channel.find("title").text
print(f"Feed title: {channel_title}")

# Find all items
items = channel.findall("item")
print(f"Number of articles: {len(items)}")

# Extract data from each item
for item in items:
    title = item.find("title").text
    category = item.find("category").text
    print(f"  [{category}] {title}")

### Key ElementTree Methods| Method | Returns | Example ||--------|---------|---------|| `find(tag)` | First matching child element | `root.find("channel")` || `findall(tag)` | List of all matching children | `channel.findall("item")` || `iter(tag)` | Iterator over all descendants | `root.iter("title")` || `.text` | Text content of an element | `element.text` || `.attrib` | Dictionary of attributes | `element.attrib` || `.tag` | Tag name | `element.tag` |Notice the similarity to BeautifulSoup from Topic 3: `find()` and `find_all()` in BeautifulSoup map to `find()` and `findall()` in ElementTree.## 4.3 XML to DataFrameLet's convert our XML articles into a DataFrame — the same JSON → DataFrame pattern applied to XML:

In [ ]:
# Extract article data into a list of dictionaries
articles = []

for item in channel.findall("item"):
    articles.append({
        "title": item.find("title").text,
        "link": item.find("link").text,
        "date": item.find("pubDate").text,
        "category": item.find("category").text,
        "description": item.find("description").text,
    })

df_articles = pd.DataFrame(articles)
df_articles

## 4.4 Real-World XML: Fetching an RSS FeedRSS feeds are one of the most common real-world XML formats. Many news sites, government bodies, and organisations publish RSS feeds. Let's fetch a live one:

In [ ]:
# Fetch a real RSS feed from the BBC
rss_url = "https://feeds.bbci.co.uk/news/technology/rss.xml"
rss_response = requests.get(rss_url, timeout=10)

print(f"Status: {rss_response.status_code}")
print(f"Content type: {rss_response.headers.get('Content-Type', 'unknown')}")

# Parse the XML
rss_root = ET.fromstring(rss_response.content)
rss_channel = rss_root.find("channel")

print(f"Feed: {rss_channel.find('title').text}")

# Extract articles
bbc_articles = []
for item in rss_channel.findall("item"):
    title_el = item.find("title")
    desc_el = item.find("description")
    date_el = item.find("pubDate")
    bbc_articles.append({
        "title": title_el.text if title_el is not None else "",
        "description": desc_el.text if desc_el is not None else "",
        "published": date_el.text if date_el is not None else "",
    })

df_bbc = pd.DataFrame(bbc_articles)
print(f"\n{len(df_bbc)} articles scraped from BBC Technology RSS")
df_bbc.head()

Notice the defensive pattern: `if title_el is not None else ""`. XML elements can be missing, and calling `.text` on `None` will crash your code. Always handle missing elements — the assessment rewards robust error handling.## 4.5 When to Expect JSON vs XMLIn practice, most modern APIs return JSON. You are more likely to encounter XML in:- **RSS/Atom feeds** — news, blogs, podcasts- **Government open data** — some ONS and EU datasets- **Financial services** — banking APIs, XBRL financial reports- **Legacy enterprise systems** — SOAP web services, older CRM/ERP systems- **Healthcare** — HL7/FHIR standards> 💼 **Assessment connection:** The rubric mentions "converting JSON/XML into a structured data format." Demonstrating that you can handle *both* formats shows stronger technical breadth. If your primary data source is JSON, consider adding an XML example (like an RSS feed relevant to your business domain) to strengthen your submission.---

---# 📖 Part 5: API Authentication & Best Practices*Independent study — work through this section in your own time*## 5.1 API AuthenticationThe Open-Meteo API is unusual in that it requires no authentication. Most real-world APIs require you to identify yourself, typically with an **API key**.### Why APIs Require Keys- **Rate limiting** — the service can track how many requests you make- **Usage billing** — some APIs charge per request- **Access control** — different keys can have different permissions- **Abuse prevention** — the service can revoke keys that misuse the API### Using API KeysAPI keys are usually passed in one of three ways:**1. As a query parameter:**```pythonparams = {"api_key": "your_key_here", "query": "London"}response = requests.get("https://api.example.com/data", params=params)```**2. As a request header:**```pythonheaders = {"Authorization": "Bearer your_key_here"}response = requests.get("https://api.example.com/data", headers=headers)```**3. As a custom header:**```pythonheaders = {"X-API-Key": "your_key_here"}response = requests.get("https://api.example.com/data", headers=headers)```> ⚠️ **Security:** Never commit API keys to Git repositories or include them directly in shared notebooks. Use environment variables or a separate configuration file that is excluded from version control.

In [ ]:
import os

# Best practice: load API keys from environment variables
# api_key = os.environ.get("MY_API_KEY")

# For coursework, you can also use a simple config file
# that you add to .gitignore:
#
# config.json:
# {"api_key": "your_key_here"}
#
# In your notebook:
# with open("config.json") as f:
#     config = json.load(f)
# api_key = config["api_key"]

print("API key loading patterns demonstrated (see code comments)")

## 5.2 Rate Limiting and Pagination### Rate LimitingMost APIs limit how many requests you can make in a given time window. If you exceed the limit, you will receive a **429 Too Many Requests** response.A robust approach handles this automatically:

In [ ]:
def api_request_with_retry(url: str, params: dict = None, max_retries: int = 3) -> dict:
    """
    Make an API request with automatic retry on rate limiting.

    Parameters
    ----------
    url : str
        The API endpoint URL
    params : dict, optional
        Query parameters
    max_retries : int
        Maximum number of retry attempts

    Returns
    -------
    dict
        Parsed JSON response
    """
    for attempt in range(max_retries):
        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            return response.json()

        elif response.status_code == 429:
            # Rate limited — wait and retry
            retry_after = int(response.headers.get("Retry-After", 5))
            print(f"Rate limited. Waiting {retry_after}s before retry {attempt + 1}/{max_retries}")
            time.sleep(retry_after)

        else:
            response.raise_for_status()

    raise Exception(f"Failed after {max_retries} retries")

# Test with a valid request
result = api_request_with_retry(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 51.5074, "longitude": -0.1278, "hourly": "temperature_2m", "forecast_days": 1}
)
print(f"Success: received {len(result['hourly']['time'])} hourly readings")

### PaginationWhen an API has a lot of data, it returns results in **pages** rather than all at once. You need to make multiple requests, advancing through the pages:```pythonall_results = []page = 1while True:    response = requests.get(url, params={"page": page, "per_page": 100})    data = response.json()    if not data["results"]:  # Empty page = we've reached the end        break    all_results.extend(data["results"])    page += 1    time.sleep(1)  # Respect rate limits```This pattern appears in many real-world APIs — the ONS API in Part 6 uses it, as do GitHub, Twitter/X, and most e-commerce APIs.## 5.3 Stretch Activity: HTTP Request/Response DebuggingWhen working with unfamiliar APIs, it can be helpful to see the full details of each request and response. You can add a **response hook** to the `requests` library that logs this information automatically:

In [ ]:
from requests import Session

def http_logger(response, *args, **kwargs):
    """Log the details of an HTTP request and response."""
    req = response.request
    url = urlparse(req.url)
    uri = "?".join([url.path, url.query]) if url.query else url.path

    print(f"\n→ {req.method} {url.hostname}{uri[:80]}{'...' if len(uri) > 80 else ''}")
    print(f"  Headers: { {k: v for k, v in list(req.headers.items())[:3]} }")

    print(f"← {response.status_code} {response.reason} ({len(response.content):,} bytes)")
    content_type = response.headers.get("Content-Type", "unknown")
    print(f"  Content-Type: {content_type}")

# Create a session with the logger attached
debug_session = Session()
debug_session.hooks["response"] = [http_logger]

# Now every request through this session gets logged
result = debug_session.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 51.5074, "longitude": -0.1278, "hourly": "temperature_2m", "forecast_days": 1}
)

This is a useful debugging technique when you are troubleshooting API issues — you can see exactly what headers were sent, what status code was returned, and how large the response body is.---

---# 📖 Part 6: End-to-End Pipeline — ONS API*Independent study — this section demonstrates the complete API → structure → analyse workflow*## 6.1 The Office for National Statistics APIThe [ONS API](https://developer.ons.gov.uk/) provides access to UK economic and demographic data — exactly the kind of public data source the assessment brief suggests. Let's build a pipeline that fetches population data, structures it, and produces a simple analysis.We will use the ONS "datasets" API to retrieve population estimates:

In [ ]:
# ONS API — fetch available datasets to explore what's available
ons_base = "https://api.beta.ons.gov.uk/v1"

response = requests.get(f"{ons_base}/datasets", timeout=15)

if response.status_code == 200:
    datasets = response.json()

    # Show the first few datasets
    print(f"Total datasets available: {datasets.get('total_count', 'unknown')}")
    print("\nFirst 5 datasets:")
    for item in datasets.get("items", [])[:5]:
        title = item.get("title", "Untitled")
        desc = item.get("description", "No description")[:80]
        print(f"  • {title}")
        print(f"    {desc}...")
        print()
else:
    print(f"Error: {response.status_code}")
    print("Note: The ONS API can be intermittent. If this fails, try again in a moment.")

## 6.2 Fetching Specific DataLet's fetch Consumer Price Inflation data — a key economic indicator that businesses monitor:

In [ ]:
# Fetch CPIH (Consumer Prices Index including owner occupiers' housing costs)
# This is a commonly used ONS dataset for economic analysis
cpih_url = f"{ons_base}/datasets/cpih01/editions"

response = requests.get(cpih_url, timeout=15)

if response.status_code == 200:
    editions = response.json()
    print("CPIH editions available:")
    for edition in editions.get("items", [])[:3]:
        print(f"  Edition: {edition.get('edition', 'unknown')}")
        links = edition.get("links", {})
        latest = links.get("latest_version", {}).get("href", "N/A")
        print(f"  Latest version: {latest}")
        print()
else:
    print(f"Error {response.status_code}: {response.text[:200]}")
    print("\nFallback: We'll work with saved sample data instead.")

### Working with the ResponseThe ONS API responses can be deeply nested. Here is a general pattern for working with any unfamiliar API:1. **Make the request** and check the status code2. **Explore the top-level keys** to understand the structure3. **Drill down** into the nested data to find what you need4. **Extract** the relevant fields into a DataFrame5. **Save** the raw response for reproducibility

In [ ]:
# General pattern for exploring any JSON API response
def explore_json(data, indent=0, max_depth=3):
    """Print the structure of a JSON response to understand its shape."""
    if indent > max_depth:
        print("  " * indent + "...")
        return

    if isinstance(data, dict):
        for key in list(data.keys())[:8]:  # Limit to first 8 keys
            value = data[key]
            if isinstance(value, dict):
                print("  " * indent + f"{key}: {{dict with {len(value)} keys}}")
                explore_json(value, indent + 1, max_depth)
            elif isinstance(value, list):
                print("  " * indent + f"{key}: [list with {len(value)} items]")
                if value and isinstance(value[0], dict):
                    explore_json(value[0], indent + 1, max_depth)
            else:
                print("  " * indent + f"{key}: {repr(value)[:60]}")
    elif isinstance(data, list) and data:
        explore_json(data[0], indent, max_depth)

# Use it on any API response
print("Structure of the weather API response:")
explore_json(data_extended, max_depth=2)

## 6.3 Connecting to Your AssessmentThe complete pipeline for your assessment looks like this:```┌─────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐│  1. Collect  │ →  │  2. Clean &  │ →  │  3. Analyse  │ →  │  4. Present  ││  via API     │    │  Structure   │    │  (NLP, etc.) │    │  Insights    │└─────────────┘    └──────────────┘    └──────────────┘    └──────────────┘```This is the same pipeline from Topic 3, but with an API as the data source instead of web scraping. In many cases, your assessment will combine both:- **Topic 3** techniques for websites without APIs- **Topic 4** techniques for services with APIs- **Topic 2** preprocessing for cleaning the collected text- **Topics 6–7** for NLP analysis and LLM-based insights> 💼 **Assessment connection:** The assessment brief specifically suggests "Find a free API and use Python to extract some information converting it from JSON/XML into a structured data format like a Pandas DataFrame." This section demonstrates exactly that pattern.---

---# 📝 Exercises## Consolidation Exercises### Exercise 1: Different Location, Different DataUse the `get_weather_forecast()` function to compare the 7-day temperature forecast for two cities (e.g. London and Edinburgh, or your workplace city and another). Create a single DataFrame with both cities and calculate the average temperature difference.### Exercise 2: JSON ExplorationChoose a free API from [this list of public APIs](https://github.com/public-apis/public-apis) (look for ones marked "No" under Auth for the easiest start). Make a request, explore the JSON structure using the `explore_json()` function, and convert the response into a DataFrame.### Exercise 3: XML to DataFrameFind an RSS feed relevant to your industry or organisation (most news sites have one — look for an RSS icon or add `/rss` or `/feed` to the URL). Parse it with ElementTree and create a DataFrame of article titles, dates, and descriptions.### Exercise 4: Error HandlingExtend the `api_request_with_retry()` function to also handle:- Connection timeouts (use `requests.exceptions.Timeout`)- Network errors (use `requests.exceptions.ConnectionError`)- Invalid JSON responsesTest it with both valid and invalid URLs.## Extension Exercises### Exercise 5: Workplace Application (Assessment Preparation)Identify an API relevant to your organisation. Write an analysis (around 300 words) covering:1. What data does the API provide, and what business question could it answer?2. What format does it return (JSON, XML, or both)?3. Does it require authentication? What are the rate limits?4. How would you structure the data for analysis?> This directly prepares you for the technical document in your summative assessment.### Exercise 6: End-to-End PipelineBuild a complete pipeline that:1. Fetches data from an API of your choice2. Handles errors and rate limits3. Converts the response to a structured DataFrame4. Saves both the raw JSON and the processed DataFrame5. Produces a simple analysis or visualisationDocument each step with clear comments explaining your decisions.### Exercise 7: Stretch — HTTP DebuggingUsing the `http_logger` hook from Part 5.3, investigate the headers sent by your requests. Modify the User-Agent header to identify your scraper and observe how different APIs respond to different User-Agent strings.---## Further Reading- [Open-Meteo API Documentation](https://open-meteo.com/en/docs)- [ONS Developer Hub](https://developer.ons.gov.uk/)- [Python requests Library Documentation](https://docs.python-requests.org/)- [Python json Module Documentation](https://docs.python.org/3/library/json.html)- [Python ElementTree Documentation](https://docs.python.org/3/library/xml.etree.elementtree.html)- [Public APIs List](https://github.com/public-apis/public-apis) — curated list of free APIs- [Automate the Boring Stuff — Chapter 16: Working with JSON Data](https://automatetheboringstuff.com/2e/chapter16/)